# ML4PAY Hackathon ITMO x AVITO
*Команда: Хог 2.6*

In [ ]:
import polars as pl
import numpy as np

import random, gc, os
from pathlib import Path

import lightgbm as lgb
from catboost import CatBoostClassifier, Pool

from sklearn.metrics import classification_report, f1_score, precision_recall_curve

import warnings
warnings.filterwarnings('ignore')

In [ ]:
RND = 67

def set_seed(seed):
    random.seed(seed)
    os.environ['PYHASHSEED'] = str(seed)
    np.random.seed(seed)

set_seed(RND)

In [ ]:
train_dir  = Path("/kaggle/input/datasets/tensonn/probablyitswork/final_dataset.parquet")
test_dir   = Path("/kaggle/input/datasets/tensonn/test-df/train_merged.parquet")
coords_dir = Path("/kaggle/input/datasets/tensonn/cities/city_coordinates.csv")
out_dir    = Path("submission.csv")

# 1) Обработка данных

### Делим данные на три части по временным периодам:
- 2025.10.01 - 2025.10.10 - Train 
- 2025.10.11 - 2025.10.20 - Test

In [ ]:
TRAIN_END_DATE = date(2025, 10, 1)
TEST_START_DATE = date(2025, 10, 11)
TEST_END_DATE = date(2025, 10, 21)

In [ ]:
train_lf = pl.scan_parquet(train_dir)
test   = pl.scan_parquet(test_dir)

def f(name, df):
    df = df.with_columns([
        pl.col("order_create_date")
        .cast(pl.Utf8)
        .str.to_datetime(strict=False),
    
        pl.col("registrationtime")
        .cast(pl.Utf8)
        .str.to_datetime(strict=False),
    
        pl.col("starttime")
        .cast(pl.Utf8)
        .str.to_datetime(strict=False),
    
        pl.col("firstlistingdate")
        .cast(pl.Utf8)
        .str.to_datetime(strict=False),
    
        pl.col("created_txtime")
        .cast(pl.Utf8)
        .str.to_datetime(strict=False),
    ])
    if name == 'train':
        df = df.filter(pl.col('order_create_date').dt.date < pl.lit(TRAIN_END_DATE))
        
    else: df = df.filter(pl.col('order_create_date') >= pl.lit(TEST_START_CREATE_DATE) &
                       pl.col('order_create_date') < pl.lit(TEST_END_CREATE_DATE))
    return df

train_lf = f('train', train_lf)
test_lf   = f('', test_lf)

In [ ]:
train_lf = train_lf.with_columns(
    pl.col("gender").fill_null("female")
)

test_lf = test_lf.with_columns(
    pl.col("gender").fill_null("female")
)

Обработка временных признаков

In [ ]:
DATE_TIME_COLS = [
    "order_create_date",
    "registrationtime",
    "starttime",
    "firstlistingdate",
    "created_txtime",
]

def fix_dates(df):
    return df.with_columns([
        
        (pl.col(col)
         .cast(pl.Utf8)
         .str.replace("T", " ")
         .str.slice(0, 19)
         .str.strptime(
                pl.Datetime,
                "%Y-%m-%d %H:%M:%S",
                strict=False
            ).alias(col))
        
        for col in DATETIME_COLS
        if col in lf.collect_schema().names()
    ])

In [ ]:
train_lf = fix_dates(train_lf)
test_lf   = fix_dates(test_lf)

### Заполняем пропуски модой

TRAIN + TEST separately

In [ ]:
fill_mode_cols = [
    "registrationtime", "starttime",
    "firstlistingdate", "platform_id",
    "payment_method", "isblocked",
    "iscompany", "category_name"
]

In [ ]:
for col in fill_mode_cols:
    col_mode_train = (
        train_lf
        .group_by(col)
        .len()
        .filter(pl.col(col).is_not_null())
        .sort("len", descending=True)
        .select(col)
        .collect()[0,0]
    )
    col_mode_test = (
        test_lf
        .groupby(col)
        .len()
        .filter(pl.col(col).is_not_null())
        .sort("len", descending=True)
        .select(col)
        .collect()[0,0]
    )

    train_lf = train_lf.with_columns(pl.col(col).fill_null(col_mode_train))
    test_lf  = test_lf.with_columns(pl.col(col).fill_null(col_mode_test))
    

Amount будет заполнять медианой внутри каждого categroy name

In [ ]:
# amount

def amount_fill(df):
    
    category_amount_median = (
        df
        .filter(pl.col("amount") != -1)
        .group_by("category_name")
        .agg(
            pl.col("amount").median().alias("category_amount_median")
        )
    )

    df = (
        train_lf
        .join(category_amount_median, on="category_name", how="left")
        .with_columns(
            pl.when(
                pl.col("amount").is_null() |
                (pl.col("amount") == -1)
            )
            .then(pl.col("category_amount_median"))
            .otherwise(pl.col("amount"))
            .alias("amount")
        )
        .drop("category_amount_median")
    )
    
    return df

train_lf = amount_fill(train_lf)
test_lf   = amount_fill(test_lf)

# 2) Генерация дополнительных признаков

**Основные доп признаки:**
1) order_weekday - день недели заказа
2) buyer_age - как быстро был создан заказ относительно регистрации покупателя на платформе
3) seller_activity_age_days - общее время нахождения продавца на платформе
4) order_price_median - медина по цене заказа 
5) microcat_median_price - медианная цена по микрокатегории 

In [ ]:
schema_cols = set(train_lf.collect_schema().names())

print(schema_cols)

In [ ]:
def get_median(lf, col):
    return lf.select(pl.col(col).median()).collect().item()

def clean_base_frame(lf: pl.LazyFrame, order_price_median: float, microcat_stats: pl.LazyFrame) -> pl.LazyFrame:

    local_cols = set(lf.collect_schema().names())
    exprs = []

    # weekday
    exprs.append(
        pl.col("order_create_date")
        .dt.weekday()
        .cast(pl.Int8)
        .alias("order_weekday")
    )

    # buyer age
    days_diff = ( pl.col("order_create_date") - pl.col("registrationtime") ).dt.total_days()
    
    exprs.append(
        pl.when(days_diff < 0)
        .then(0)
        .otherwise(days_diff)
        .fill_null(0)
        .cast(pl.Int32)
        .alias("buyer_account_age_days")
    )
    
    # seller activity age
    seller_days_diff = ( pl.col("firstlistingdate") - pl.col("registrationtime") ).dt.total_days()

    exprs.append(
        pl.when(seller_days_diff < 0)
        .then(0)
        .otherwise(seller_days_diff)
        .fill_null(0)
        .cast(pl.Int32)
        .alias("seller_activity_age_days")
    )

    # order price
    price_filled = (pl.col("order_price").fill_null(order_price_median))

    exprs.append(
        pl.when(price_filled < 0)
        .then(0)
        .otherwise(price_filled)
        .alias("order_price")
    )

    return lf.with_columns(exprs)

In [ ]:
order_price_median = get_median(train_lf, "order_price")

microcat_stats = train_lf.group_by("microcat_name").agg(
    pl.col("order_price").median().alias("microcat_median_price")
)


train_lf = clean_base_frame(
    train_lf,
    order_price_median,
    microcat_stats
)

test_lf = clean_base_frame(
    test_lf,
    order_price_median,
    microcat_stats
)

**Дополнительные признаки на основе координат городов (собранные данные через geopy)**

In [ ]:
coords_df = pl.read_csv(coords_dir)

coords_df = coords_df.with_columns(
    pl.col('lat').is_not_null() & pl.col('lon').is_not_null()
)

coords_df.head(5)

**Расстояние до центра кластера которому принадлежит город**

Распределим города на кластеры с помощью KMeans, а затем будет добовлять расстояние до центра класетра как дополнительный признак  

In [ ]:
n_clusters = 50

coords_pd = coords_df.select(['city', 'lon', 'lat']).to_pandas()
X_k = coords_pd[['lon', 'lat']].to_numpy()

In [ ]:
knn = KMeans(n_clusters = n_clusters,random_state = RND)

coords_pd['geo_cluster'] = knn.fit_predict(X_k)

centers = knn.cluster_centers_

In [ ]:
# Расчитаем расстояние до центров через haversine

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))
    

In [ ]:
coords_pd["geo_cluster_center_lat"] = coords_pd["geo_cluster"].map(lambda c: centers[c, 0])
coords_pd["geo_cluster_center_lon"] = coords_pd["geo_cluster"].map(lambda c: centers[c, 1])

coords_pd["geo_dist_to_cluster_center_km"] = haversine_km(
    coords_pd["lat"].to_numpy(),
    coords_pd["lon"].to_numpy(),
    coords_pd["geo_cluster_center_lat"].to_numpy(),
    coords_pd["geo_cluster_center_lon"].to_numpy()
)

coords_feat = pl.from_pandas(
    coords_pd[[
        "city",
        "geo_cluster",
        "geo_dist_to_cluster_center_km"
    ]]
)

coords_feat.head(5)

In [ ]:
train_lf = train_lf.join(coords_feat.lazy(), on='city', how='left')
test_lf  = test_lf.join(coords_feat.lazy(),  on='city', how='left')

In [ ]:
train_lf = train_lf.with_columns([
    pl.col("geo_cluster").fill_null(-1).cast(pl.Int32),
    pl.col("geo_dist_to_cluster_center_km").fill_null(-1.0),
])

test_lf = test_lf.with_columns([
    pl.col("geo_cluster").fill_null(-1).cast(pl.Int32),
    pl.col("geo_dist_to_cluster_center_km").fill_null(-1.0),
])

In [ ]:
train_lf = train_lf.drop("city")
test_lf = test_lf.drop("city")

# 3) Обучение модели

In [ ]:
# подготовка данных

df_train = train_lf.collect().to_pandas()
target_col = 'is_return'

y = df_train[target_col].astype(int).values()
X = df_train.drop(columns=[target_col])

In [ ]:
for col in X.columns:
    if pd.api.types.is_datetime64_any_dtype(X[col]):
        X[col] = X[col].astype("int64") // 10**9

bool_cols = X.select_dtypes(include=["bool"]).columns.tolist()
for col in bool_cols:
    X[col] = X[col].astype(int)

In [ ]:
cat_cols = []
text_cols = []

for col in cat_cols:
    X[col] = X[col].astype('category').cat.add_categroy('MISSING')
    X[col] = X[col].fillna('MISSING')

for col in text_cols:
    X[col] = X[col].fillna('MISSING').astype(str)

In [ ]:
n_folds = 3
oof_pred = np.zeros(len(X), dtype=float)
fold_pr_auc = []

skf = StratifiedKFold(n_splits = n_folds, shuffle = True, random_state = RND)
bar = tqdm(skf.split(X, y))

for fold, (tr_idx, va_idx) in enumerate(bar):
    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_va, y_va = X[va_idx], y[va_idx]

    params = {
        'random_seed': RND + fold,
        'learning_rate': 0.05,
        'auto_class_weights': "SqrtBalanced",
        'eval_metric': 'AUC',
        'task_type': 'GPU',
        'l2_leaf_reg': 5,
        'iterations': 2000, 
        'depth': 8,
        'early_stopping_rounds': 200,
    }

    print(params)
    
    tr_pool = Pool(X_tr, y_tr, cat_features = cat_cols, text_features = text_cols)
    va_pool = Pool(X_va, y_va, cat_features = cat_cols, text_features = text_cols)

    model = CatBoostClassifier(**params)
    model.fit(tr_pool, eval_set = va_pool, verbose = 200)

    va_pred = model.predict_proba(X_va)[:, 1]
    oof_pred[fold] = val_pred

    pr_auc = precision_recall_curve(y_va, va_pred)
    fold_pr_auc.append(pr_auc)
    bar.set_postfix(f'VAL PR_AUC = {pr_auc}')

In [15]:
oof_pr_auc = average_precision_score(y, oof_pred)

precision, recall, thresholds = precision_recall_curve(y, oof_pred)
f1_vals = 2 * precision * recall / (precision + recall + 1e-12)

best_idx = np.argmax(f1_vals)
best_threshold = thresholds[max(best_idx - 1, 0)] if len(thresholds) > 0 else 0.5

y_pred = (oof_pred >= best_threshold).astype(int)

In [16]:
print("=== OOF RESULTS ===")

print(f"OOF PR AUC: {oof_pr_auc:.6f}")
print(f"Mean fold PR AUC: {np.mean(fold_pr_auc):.6f} ± {np.std(fold_pr_auc):.6f}")
print(f"Best threshold: {best_threshold:.6f}")
print(f"Precision: {precision_score(y, y_pred, zero_division=0):.6f}")
print(f"Recall:    {recall_score(y, y_pred, zero_division=0):.6f}")
print(f"F1:        {f1_score(y, y_pred, zero_division=0):.6f}")

Help on class Pool in module catboost.core:

class Pool(_catboost._PoolBase)
 |  Pool(data, label=None, cat_features=None, text_features=None, embedding_features=None, embedding_features_data=None, column_description=None, pairs=None, graph=None, delimiter='\t', has_header=False, ignore_csv_quoting=False, weight=None, group_id=None, group_weight=None, subgroup_id=None, pairs_weight=None, baseline=None, timestamp=None, feature_names=None, feature_tags=None, thread_count=-1, log_cout=None, log_cerr=None, data_can_be_none=False)
 |
 |  Pool used in CatBoost as a data structure to train model from.
 |
 |  Method resolution order:
 |      Pool
 |      _catboost._PoolBase
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, data, label=None, cat_features=None, text_features=None, embedding_features=None, embedding_features_data=None, column_description=None, pairs=None, graph=None, delimiter='\t', has_header=False, ignore_csv_quoting=False, weight=None, group_id=None, g

In [ ]:
params = {
    'random_seed': RND,
    'learning_rate': 0.05,
    'auto_class_weights': "SqrtBalanced",
    'eval_metric': 'AUC',
    'task_type': 'GPU',
    'l2_leaf_reg': 5,
    'iterations': 2000, 
    'depth': 8,
    'early_stopping_rounds': 200,
}


final_model = CatBoostClassifier(**params)
final_model.fit(
    X, 
    y, 
    cat_features=cat_cols, 
    verbose=200
)

# 4) Предсказание

In [ ]:
test_df = test_lf.collect().to_pandas()

submission_item_ids = test_item_ids.reset_index(drop=True)
X_test = test_df.drop(columns=["is_return"], errors="ignore").copy()

for col in X_test.columns:
    if pd.api.types.is_datetime64_any_dtype(X_test[col]):
        X_test[col] = X_test[col].astype("int64") // 10**9

bool_cols = X_test.select_dtypes(include=["bool"]).columns.tolist()

for col in bool_cols:
    X_test[col] = X_test[col].astype(int)


for col in cat_cols:
    X[col] = X[col].astype('category').cat.add_categroy('MISSING')
    X[col] = X[col].fillna('MISSING')

for col in text_cols:
    X[col] = X[col].fillna('MISSING').astype(str)

In [ ]:
test_proba = final_model.predict_proba(X_test)[:, 1]

test_pred = (test_proba >= best_threshold).astype(int)

submission = pd.DataFrame({
    "item_id": submission_item_ids,
    "is_return": test_pred,
})

submission.to_csv(
    out_dir,
    index=False
)

submission.head()